In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

import os, sys
from pathlib import Path
import functools

from extra.components import XrayPulses
import extra_data as ex
from extra_data.components import AGIPD1M
from extra_geom import AGIPD_1MGeometry
from extra_data import by_index
import extra_speckle as esp
from extra_speckle.setup import Setup

from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import dask.array as da

import numpy as np
import matplotlib.pyplot as plt
import h5py
from pyFAI.integrator.azimuthal import AzimuthalIntegrator

In [23]:
from analysis.agipd_integrate import NPT, UNIT, _ai, integrate_part

PROPOSAL, RUN_NO = 10400, 423
TRAINS_PER_PART = 2
OUT = f"/home/andronis/MID-10400/proc/r{RUN_NO:04d}_saxs.h5"

In [3]:
cluster = SLURMCluster(
    queue="upex",
    cores=1,
    processes=1,
    memory="32GB",
    local_directory="/scratch",
    job_extra_directives=["--mem=32G"],
)
cluster.scale(25)
client = Client(cluster)
client.wait_for_workers(25, timeout=600)

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at: tcp://131.169.183.130:40209
INFO:distributed.scheduler:  dashboard at:  http://131.169.183.130:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.scheduler:Receive client connection: Client-ccb3acf6-aa34-11f1-85ea-a963fb00c97a
INFO:distributed.core:Starting established connection to tcp://131.169.183.130:35164
INFO:distributed.scheduler:Register worker addr: tcp://131.169.178.170:37557 name: SLURMCluster-19
INFO:distributed.scheduler:Starting worker compute stream, tcp://131.169.178.170:37557
INFO:distributed.core:Starting established connection to tcp://131.169.178.170:60918
INFO:distributed.scheduler:Register worker addr: tcp://131.169.178.154:41899 name: SLURMCluster-15
INFO:distributed.scheduler:Starting worker com

In [24]:
det = AGIPD1M(ex.open_run(PROPOSAL, RUN_NO, data="proc"), min_modules=16)
n_trains = len(det.train_ids)
slices = [
    slice(i, min(i + TRAINS_PER_PART, n_trains))
    for i in range(0, n_trains, TRAINS_PER_PART)
]

In [25]:
futures = client.map(
    integrate_part, [PROPOSAL] * len(slices), [RUN_NO] * len(slices), slices
)
parts = client.gather(futures)

train_id = np.concatenate([p[0] for p in parts])
pulse_id = np.concatenate([p[1] for p in parts])
intensity = np.concatenate([p[2] for p in parts])

order = np.lexsort((pulse_id, train_id))
train_id, pulse_id, intensity = train_id[order], pulse_id[order], intensity[order]

KeyboardInterrupt: 

In [ ]:
_, q, _ = _ai()

In [ ]:
from analysis.agipd_integrate import GEOM_PATH, MASK_PATH, SDD_MM, WAVELENGTH_M

with h5py.File(OUT, "w") as f:
    f.create_dataset("q", data=q)
    f.create_dataset("trainId", data=train_id)
    f.create_dataset("pulseId", data=pulse_id)
    f.create_dataset(
        "I",
        data=intensity,
        chunks=(min(256, len(intensity)), NPT),
        compression="gzip",
        compression_opts=1,
    )
    f["q"].attrs["units"] = UNIT
    f.attrs["run"] = RUN_NO
    f.attrs["geom"] = GEOM_PATH
    f.attrs["mask"] = MASK_PATH
    f.attrs["npt"] = NPT
    f.attrs["sdd_mm"] = SDD_MM
    f.attrs["wavelength_m"] = WAVELENGTH_M
    f.attrs["center_xy"] = (607.4598195630211, 672.076693118667)